# Sequential Pattern Mining Toolkit: AprioriAll & SPMF Integration

This interactive notebook serves as a supplementary demonstration for the accompanying software article. It verifies the algorithmic execution pipeline, functional correctness, and performance of the Python-based AprioriAll implementations (sequential and parallel) against the native SPMF reference implementations.

## 1. Algorithmic Execution Pipeline (Demonstration Example from the Paper)

This section reproduces the step-by-step example detailed in the article to visualize the transformation states of the sequence database. The execution pipeline follows these explicit phases:
* **Step 1:** Original Sequence Database ($D_S$)
* **Step 2:** Mining frequent sets ($\mathcal{L} = \{L_1, L_2\}$ via Apriori loop)
* **Step 3:** Transformation and mapping to natural numbers ($D_{TS}$)
* **Step 4:** Sequencing (Candidate generation via `apriori_generate` join/prune loops)
* **Step 5:** Maximization (Filtering non-maximal sequential patterns)

In [1]:
# Execute the verbose test script to trace the database transformation states
!python utils/evaluation/test_apriori_all.py




==================== STEP 1: Original Sequence Database (Ds) [SEQUENTIAL MODE] ====================
Sequence ID | Sequence Layout    
------------+--------------------
1           | {1,2} -> {3} -> {4}
2           | {1} -> {4,5}       
3           | {1,2} -> {4}       
4           | {2} -> {3} -> {4}  

==================== STEP 2: Frequent Itemsets Mining (L) ====================
Itemset ID | Frequent Itemset Layout
-----------+------------------------
1          | {1}                    
2          | {2}                    
3          | {3}                    
4          | {4}                    
5          | {1,2}                  

==================== STEP 3: Transformed Mapped Database (Dts) ====================
Sequence ID | Transformed (Mapped IDs)
------------+-------------------------
1           | {1,2} -> {3} -> {4}     
2           | {1} -> {4,5}            
3           | {1,2} -> {4}            
4           | {2} -> {3} -> {4}       

==================== STEP 4: Seque

## 2. Data Preparation

This AprioriAll implementation uses a more human-readable format than the default SPMF dataset format, thus a conversion is done.

In [2]:
# Generate the sequence file for AprioriAll
!python utils/convert_spmf_to_seq.py --input data/raw/sign.txt --out data/seq_sign.csv --direction spmf2csv
print("\n Dataset converted to CSV format: data/seq_sign.csv")

OK: Saved to data/seq_sign.csv (sequences=730)

 Dataset converted to CSV format: data/seq_sign.csv


## 3. Functional Verification against Reference Implementations

To verify the correctness of the AprioriAll implementation, cross-validation is performed against the native SPMF execution layer using the sign dataset (5,834 sequences). For any given minimum support threshold (min_sup_count, representing the absolute number of occurrences), both independent codebases must discover the exact same number of frequent patterns.

In [3]:
import pandas as pd

csv_dataset = "data/seq_sign.csv"
raw_dataset = "data/raw/sign.txt"
min_sup_count = 200

print("Executing Python-based AprioriAll pipeline... (please wait)", flush=True)
# 1. Execute Python-based AprioriAll pipeline on the CSV
!python -u utils/evaluation/run_pipeline.py --algo apriori_all --input {csv_dataset} --min-sup-count {min_sup_count} --out-dir output

print("Executing SPMF PrefixSpan reference implementation... (please wait)", flush=True)
# 2. Execute SPMF PrefixSpan reference implementation on the source file
!python -u utils/evaluation/run_pipeline.py --algo prefixspan --input {raw_dataset} --min-sup-count {min_sup_count} --out-dir output

print("Loading and cross-verifying pattern cardinalities...", flush=True)
# 3. Load and cross-verify pattern cardinalities
python_patterns = pd.read_csv('output/frequent_sequences.csv')
spmf_patterns = pd.read_csv('output/frequent_sequences.txt', sep=' #SUP: ', engine='python', header=None)

print(f"\n[VERIFICATION SUMMARY]")
print(f"AprioriAll pattern count: {len(python_patterns)}")
print(f"PrefixSpan pattern count:  {len(spmf_patterns)}")

if len(python_patterns) == len(spmf_patterns):
    print("SUCCESS: Implementations are functionally identical and outputs match perfectly.")
else:
    print("ERROR: Mismatch detected in final pattern cardinalities.")

Executing Python-based AprioriAll pipeline... (please wait)
=== RUNNING APRIORI ALL ALGORITHM ===
Total sequences loaded: 730
Minimum Support Count threshold: 200 (27.40%)

-> [STEP] Step 1: Sorted and Grouped Sequences Database (Ds)                | Total Elements: 730
-> [STEP] Step 2: Discovered Frequent Itemsets (L)                          | Total Elements: 80
-> [STEP] Step 3: Frequent Itemsets Mapping to Integers                     | Total Elements: 80
-> [STEP] Step 3: Transformed and Mapped Database (Dts)                     | Total Elements: 730
-> [STEP] Step 4: Frequent 1-Sequences (LS1)                                | Total Elements: 80
-> [STEP] Step 4: Candidate 2-Sequences (CS2)                               | Total Elements: 6400
-> [STEP] Step 4: Frequent 2-Sequences (LS2)                                | Total Elements: 600
-> [STEP] Step 4: Candidate 3-Sequences (CS3)                               | Total Elements: 3140
-> [STEP] Step 4: Frequent 3-Sequences (LS3)

## 4. Scalability and Performance Benchmarking

The runtime characteristics of the framework are evaluated by executing the benchmark suite.
To maintain strict input boundaries:
1. The Python implementations (`apriori_all`, `apriori_all_parallel`) are run using the converted `.csv` file.
2. The SPMF algorithms (`prefixspan`, `gsp`) are run using the source `.txt` file, appending the results to the same benchmark log.

In [ ]:
print("Benchmarking Python implementations... This might take several minutes.", flush=True)
# 1. Benchmark Python implementations using the converted CSV file
# The --clear-results flag is used to start a fresh benchmark log
!python -u utils/evaluation/run_benchmarks.py --input data/seq_sign.csv --supports 120 160 200 --algos apriori_all apriori_all_parallel --out-dir output --clear-results

print("\nBenchmarking SPMF implementations... Appending to log.", flush=True)
# 2. Benchmark SPMF implementations using the source file
# The --clear-results flag is omitted so these metrics are appended to the same log file
!python -u utils/evaluation/run_benchmarks.py --input data/raw/sign.txt --supports 120 160 200 --algos prefixspan gsp --out-dir output

print("\nBenchmark suite finished!", flush=True)

## 5. Performance Analysis and Metrics

This section loads the compiled performance log metrics from the benchmark suite. The data is presented in a tabular format for exact metric inspection, featuring absolute and relative support thresholds, and plotted to construct execution runtime curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Load the generated benchmark metrics
csv_path = 'output/benchmarks/benchmark_results.csv'

try:
    df = pd.read_csv(csv_path)
    
    # Extract dataset metadata dynamically from the logs
    dataset_name = df['input'].iloc[0] if not df.empty else "Unknown Dataset"
    total_sequences = df['input_sequences'].iloc[0] if not df.empty else 1
    
    # Calculate minimum support as a percentage of the dataset
    df['support_percent'] = (df['min_sup_count'] / total_sequences) * 100
    
    # Sort for consistent presentation (by algorithm name, then by support descending)
    df = df.sort_values(by=['algo', 'support_percent'], ascending=[True, False]).reset_index(drop=True)
    
    # Prepare a formatted copy for the display table
    display_df = df[['algo', 'min_sup_count', 'support_percent', 'pattern_count', 'duration_sec', 'max_memory_mb', 'avg_cpu_percent']].copy()
    
    # Rename columns to human-readable format
    display_df.columns = [
        'Algorithm', 
        'Min Support (Abs)', 
        'Min Support (%)', 
        'Patterns Found', 
        'Execution Time (s)', 
        'Peak RAM (MB)', 
        'Avg CPU (%)'
    ]
    
    # Apply string formatting for better readability
    display_df['Min Support (%)'] = display_df['Min Support (%)'].map('{:.2f}%'.format)
    display_df['Execution Time (s)'] = display_df['Execution Time (s)'].map('{:.4f}'.format)
    display_df['Peak RAM (MB)'] = display_df['Peak RAM (MB)'].map('{:.2f}'.format)
    display_df['Avg CPU (%)'] = display_df['Avg CPU (%)'].map('{:.1f}%'.format)
    
    print(f"=== BENCHMARK METRICS ===")
    print(f"Dataset: {dataset_name} | Total Sequences: {total_sequences}")
    display(display_df)

    # Generate Runtime Performance Plot using percentage on X-axis
    plt.figure(figsize=(10, 6))
    
    for algo in df['algo'].unique():
        subset = df[df['algo'] == algo]
        plt.plot(
            subset['support_percent'], 
            subset['duration_sec'], 
            marker='o', 
            linewidth=2, 
            label=algo
        )

    # Invert X-axis so stricter thresholds (lower support %) appear on the right side
    plt.gca().invert_xaxis() 
    
    plt.title(f'Performance Scaling Evaluation on Dataset: {dataset_name}')
    plt.xlabel('Minimum Support Threshold (%)')
    plt.ylabel('Execution Time (seconds)')
    plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Error: Benchmark results file not found at {csv_path}. Please execute the benchmark suite first.")
except Exception as e:
    print(f"An error occurred while processing the results: {e}")

## 6. Comprehensive Multi-Dataset Benchmarking

This final section executes the complete benchmark suite across multiple SPMF datasets to evaluate the framework's scalability and performance under varying data distributions and sequence lengths.

In [ ]:
import os
from IPython import get_ipython

benchmark_config = {
    "sign": {
        "csv": "data/seq_sign.csv",
        "txt": "data/raw/sign.txt",
        "supports": ["120", "160", "200"]
    },
    "leviathan": {
        "csv": "data/seq_leviathan.csv",
        "txt": "data/raw/leviathan.txt",
        "supports": ["583", "875", "1167"]
    },
    "bike": {
        "csv": "data/seq_bike.csv",
        "txt": "data/raw/bike.txt",
        "supports": ["210", "420", "840"]
    },
    "bible": {
        "csv": "data/seq_bible.csv",
        "txt": "data/raw/bible.txt",
        "supports": ["1818", "2727", "3637"]
    },
    "bmswebview1": {
        "csv": "data/seq_bmswebview1.csv",
        "txt": "data/raw/bmswebview1.txt",
        "supports": ["300", "600", "900"]
    }
}

out_dir = "output/benchmarks"

python_algos = "apriori_all apriori_all_parallel"
spmf_algos = "prefixspan gsp spade"

print("Starting comprehensive multi-dataset benchmark suite...\n", flush=True)

clear_flag = "--clear-results"

for dataset_name, config in benchmark_config.items():
    print(f"--- Dataset: {dataset_name.upper()} ---", flush=True)
    
    csv_file = config["csv"]
    txt_file = config["txt"]
    supports = " ".join(config["supports"])
    
    print(f"Benchmarking Python implementations for {dataset_name}...", flush=True)
    cmd_python = f"python -u utils/evaluation/run_benchmarks.py --input {csv_file} --supports {supports} --algos {python_algos} --out-dir {out_dir} {clear_flag}"
    get_ipython().system(cmd_python)
        
    clear_flag = "" 
    
    print(f"Benchmarking SPMF implementations for {dataset_name}...", flush=True)
    cmd_spmf = f"python -u utils/evaluation/run_benchmarks.py --input {txt_file} --supports {supports} --algos {spmf_algos} --out-dir {out_dir}"
    get_ipython().system(cmd_spmf)
    
    print("\n", flush=True)

print("Benchmark suite completed.", flush=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
from IPython.display import display

csv_path = 'output/benchmarks/benchmark_results.csv'

try:
    df = pd.read_csv(csv_path)
    
    df['dataset_name'] = df['input'].apply(lambda x: x.split('/')[-1].split('.')[0].replace('seq_', '').replace('raw/', ''))
    
    df['support_percent'] = (df['min_sup_count'] / df['input_sequences']) * 100
    
    display_df = df[['dataset_name', 'algo', 'support_percent', 'pattern_count', 'duration_sec', 'max_memory_mb']].copy()
    display_df.columns = ['Dataset', 'Algorithm', 'Support (%)', 'Patterns', 'Time (s)', 'Peak RAM (MB)']
    display_df['Support (%)'] = display_df['Support (%)'].map('{:.2f}%'.format)
    
    print("=== SUMMARY METRICS ===")
    display(display_df)

    datasets = df['dataset_name'].unique()
    fig, axes = plt.subplots(math.ceil(len(datasets)/2), 2, figsize=(14, 4 * math.ceil(len(datasets)/2)))
    axes = axes.flatten()
    
    for i, ds in enumerate(datasets):
        subset = df[df['dataset_name'] == ds]
        for algo in subset['algo'].unique():
            algo_data = subset[subset['algo'] == algo].sort_values('support_percent')
            axes[i].plot(algo_data['support_percent'], algo_data['duration_sec'], marker='o', label=algo)
        
        axes[i].set_title(f"Performance: {ds.capitalize()}")
        axes[i].invert_xaxis()
        axes[i].set_xlabel("Support %")
        axes[i].set_ylabel("Time (s)")
        axes[i].legend()
        axes[i].grid(True)

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("Benchmark results not found. Run Section 6 benchmarks first.")